In [1]:
"""
Python script to get Last Traded Price (LTP) of Indian stocks using Yahoo Finance.

Requirements:
    pip install yfinance

Note:
    - NSE stocks: Use format 'SYMBOL.NS' (e.g., 'INFY.NS')
    - BSE stocks: Use format 'SYMBOL.BO' (e.g., 'RELIANCE.BO')
"""



from typing import List, Dict, Optional
import pandas as pd


class StockLTP:
    """Class to fetch Last Traded Price for Indian stocks using Yahoo Finance."""
    
    def __init__(self):
        """Initialize StockLTP class."""
        pass
    
    def get_ltp(self, symbols: List[str]) -> Dict[str, Dict]:
        """
        Get Last Traded Price for given stock symbols.
        
        Args:
            symbols: List of stock symbols in Yahoo Finance format
                    NSE: 'SYMBOL.NS' (e.g., 'INFY.NS', 'SBIN.NS')
                    BSE: 'SYMBOL.BO' (e.g., 'RELIANCE.BO')
        
        Returns:
            Dictionary with symbol as key and LTP data as value
        """
        result = {}
        for symbol in symbols:
            try:
                ticker = yf.Ticker(symbol)
                info = ticker.info
                hist = ticker.history(period="1d")
                
                if not hist.empty:
                    last_price = hist['Close'].iloc[-1]
                    result[symbol] = {
                        'last_price': round(last_price, 2),
                        'open': round(hist['Open'].iloc[-1], 2) if 'Open' in hist.columns else None,
                        'high': round(hist['High'].iloc[-1], 2) if 'High' in hist.columns else None,
                        'low': round(hist['Low'].iloc[-1], 2) if 'Low' in hist.columns else None,
                        'volume': int(hist['Volume'].iloc[-1]) if 'Volume' in hist.columns else None,
                        'name': info.get('longName', symbol),
                        'currency': info.get('currency', 'INR')
                    }
                else:
                    result[symbol] = {'error': 'No data available'}
            except Exception as e:
                result[symbol] = {'error': str(e)}
        
        return result
    
    def get_ltp_single(self, symbol: str) -> Optional[float]:
        """
        Get LTP for a single stock.
        
        Args:
            symbol: Stock symbol in Yahoo Finance format
                   NSE: 'SYMBOL.NS' (e.g., 'INFY.NS')
                   BSE: 'SYMBOL.BO' (e.g., 'RELIANCE.BO')
        
        Returns:
            Last Traded Price as float, or None if error
        """
        try:
            ticker = yf.Ticker(symbol)
            hist = ticker.history(period="1d")
            
            if not hist.empty:
                return round(hist['Close'].iloc[-1], 2)
            return None
        except Exception as e:
            print(f"Error fetching LTP for {symbol}: {e}")
            return None
    
    def get_quote(self, symbol: str) -> Dict:
        """
        Get detailed quote data including LTP, OHLC, volume, etc.
        
        Args:
            symbol: Stock symbol in Yahoo Finance format
        
        Returns:
            Dictionary with detailed quote data
        """
        try:
            ticker = yf.Ticker(symbol)
            info = ticker.info
            hist = ticker.history(period="1d")
            
            if hist.empty:
                return {'error': 'No data available'}
            
            quote_data = {
                'symbol': symbol,
                'name': info.get('longName', symbol),
                'last_price': round(hist['Close'].iloc[-1], 2),
                'open': round(hist['Open'].iloc[-1], 2) if 'Open' in hist.columns else None,
                'high': round(hist['High'].iloc[-1], 2) if 'High' in hist.columns else None,
                'low': round(hist['Low'].iloc[-1], 2) if 'Low' in hist.columns else None,
                'close': round(hist['Close'].iloc[-1], 2),
                'volume': int(hist['Volume'].iloc[-1]) if 'Volume' in hist.columns else None,
                'currency': info.get('currency', 'INR'),
                'market_cap': info.get('marketCap'),
                'previous_close': info.get('previousClose'),
                'day_change': round(hist['Close'].iloc[-1] - hist['Open'].iloc[-1], 2) if len(hist) > 0 else None,
                'day_change_percent': round(((hist['Close'].iloc[-1] - hist['Open'].iloc[-1]) / hist['Open'].iloc[-1]) * 100, 2) if len(hist) > 0 and hist['Open'].iloc[-1] != 0 else None
            }
            
            return quote_data
        except Exception as e:
            return {'error': str(e)}
    
    def get_realtime_price(self, symbol: str) -> Optional[float]:
        """
        Get real-time or latest available price.
        
        Args:
            symbol: Stock symbol in Yahoo Finance format
        
        Returns:
            Current price as float, or None if error
        """
        try:
            ticker = yf.Ticker(symbol)
            data = ticker.history(period="1d", interval="1m")
            
            if not data.empty:
                return round(data['Close'].iloc[-1], 2)
            else:
                # Fallback to daily data
                return self.get_ltp_single(symbol)
        except Exception as e:
            print(f"Error fetching real-time price for {symbol}: {e}")
            return None


def main():
    """Example usage of StockLTP class."""
    
    # Initialize the StockLTP class
    stock_ltp = StockLTP()
    
    # Example 1: Get LTP for multiple stocks
    print("=== Getting LTP for multiple stocks ===")
    symbols = ['INFY.NS', 'SBIN.NS', 'TCS.NS', 'RELIANCE.NS']
    ltp_data = stock_ltp.get_ltp(symbols)
    
    for symbol, data in ltp_data.items():
        if 'error' in data:
            print(f"{symbol}: Error - {data['error']}")
        else:
            print(f"{symbol} ({data.get('name', 'N/A')}): ₹{data.get('last_price', 'N/A')}")
    
    print("\n=== Getting LTP for a single stock ===")
    # Example 2: Get LTP for a single stock
    ltp = stock_ltp.get_ltp_single('INFY.NS')
    print(f"INFY.NS LTP: ₹{ltp}")
    
    print("\n=== Getting detailed quote ===")
    # Example 3: Get detailed quote
    quote = stock_ltp.get_quote('INFY.NS')
    if 'error' not in quote:
        print(f"Symbol: {quote.get('symbol')}")
        print(f"Name: {quote.get('name')}")
        print(f"Last Price: ₹{quote.get('last_price')}")
        print(f"Open: ₹{quote.get('open')}")
        print(f"High: ₹{quote.get('high')}")
        print(f"Low: ₹{quote.get('low')}")
        print(f"Close: ₹{quote.get('close')}")
        print(f"Volume: {quote.get('volume')}")
        print(f"Day Change: ₹{quote.get('day_change')} ({quote.get('day_change_percent')}%)")
    else:
        print(f"Error: {quote.get('error')}")
    
    print("\n=== Popular Indian Stocks ===")
    # Example 4: Popular Indian stocks
    popular_stocks = {
        'NSE': ['NIFTYBEES.NS',
'INDUSINDBK.NS',
'ITC.NS',
'TMPV.NS',
'NATCOPHARM.NS',
'DRREDDY.NS',
'NEXT50IETF.NS',
'GOLDIETF.NS',
'TCS.NS',
'TATACAP.NS',
'KTKBANK.NS',
'JYOTHYLAB.NS',
'TMCV.NS',
'ZYDUSLIFE.NS',
'FINCABLES.NS',
'IDFCFIRSTB.NS',
'INFY.NS',
'AXISBANK.NS',
'WIPRO.NS',
'SPANDANA.NS',
'SOUTHBANK.NS',
'TMB.NS',
'HCLTECH.NS',
'PETRONET.NS',
'ITBEES.NS',
'YESBANK.NS',
'MANAPPURAM.NS',
'ABCAPITAL.NS',
'GOLDCASE.NS'],
        'BSE': ['RELIANCE.BO', 'TCS.BO']
    }
    
    for exchange, stocks in popular_stocks.items():
        print(f"\n{exchange} Stocks:")
        for stock in stocks:
            ltp = stock_ltp.get_ltp_single(stock)
            if ltp:
                print(f"  {stock}: ₹{ltp}")


if __name__ == "__main__":
    main()


=== Getting LTP for multiple stocks ===
INFY.NS: Error - name 'yf' is not defined
SBIN.NS: Error - name 'yf' is not defined
TCS.NS: Error - name 'yf' is not defined
RELIANCE.NS: Error - name 'yf' is not defined

=== Getting LTP for a single stock ===
Error fetching LTP for INFY.NS: name 'yf' is not defined
INFY.NS LTP: ₹None

=== Getting detailed quote ===
Error: name 'yf' is not defined

=== Popular Indian Stocks ===

NSE Stocks:
Error fetching LTP for NIFTYBEES.NS: name 'yf' is not defined
Error fetching LTP for INDUSINDBK.NS: name 'yf' is not defined
Error fetching LTP for ITC.NS: name 'yf' is not defined
Error fetching LTP for TMPV.NS: name 'yf' is not defined
Error fetching LTP for NATCOPHARM.NS: name 'yf' is not defined
Error fetching LTP for DRREDDY.NS: name 'yf' is not defined
Error fetching LTP for NEXT50IETF.NS: name 'yf' is not defined
Error fetching LTP for GOLDIETF.NS: name 'yf' is not defined
Error fetching LTP for TCS.NS: name 'yf' is not defined
Error fetching LTP for T

In [2]:
"""
Python script to get Last Traded Price (LTP) of Indian stocks using Yahoo Finance.

Requirements:
    pip install yfinance

Note:
    - NSE stocks: Use format 'SYMBOL.NS' (e.g., 'INFY.NS')
    - BSE stocks: Use format 'SYMBOL.BO' (e.g., 'RELIANCE.BO')
"""

import yfinance as yf
from typing import List, Dict, Optional
import pandas as pd


class StockLTP:
    """Class to fetch Last Traded Price for Indian stocks using Yahoo Finance."""
    
    def __init__(self):
        """Initialize StockLTP class."""
        pass
    
    def get_ltp(self, symbols: List[str]) -> Dict[str, Dict]:
        """
        Get Last Traded Price for given stock symbols.
        
        Args:
            symbols: List of stock symbols in Yahoo Finance format
                    NSE: 'SYMBOL.NS' (e.g., 'INFY.NS', 'SBIN.NS')
                    BSE: 'SYMBOL.BO' (e.g., 'RELIANCE.BO')
        
        Returns:
            Dictionary with symbol as key and LTP data as value
        """
        result = {}
        for symbol in symbols:
            try:
                ticker = yf.Ticker(symbol)
                info = ticker.info
                hist = ticker.history(period="2d")
                
                if not hist.empty:
                    last_price = hist['Close'].iloc[-1]
                    result[symbol] = {
                        'last_price': round(last_price, 2),
                        'open': round(hist['Open'].iloc[-1], 2) if 'Open' in hist.columns else None,
                        'high': round(hist['High'].iloc[-1], 2) if 'High' in hist.columns else None,
                        'low': round(hist['Low'].iloc[-1], 2) if 'Low' in hist.columns else None,
                        'volume': int(hist['Volume'].iloc[-1]) if 'Volume' in hist.columns else None,
                        'name': info.get('longName', symbol),
                        'currency': info.get('currency', 'INR')
                    }
                else:
                    result[symbol] = {'error': 'No data available'}
            except Exception as e:
                result[symbol] = {'error': str(e)}
        
        return result
    
    def get_ltp_single(self, symbol: str) -> Optional[float]:
        """
        Get LTP for a single stock.
        
        Args:
            symbol: Stock symbol in Yahoo Finance format
                   NSE: 'SYMBOL.NS' (e.g., 'INFY.NS')
                   BSE: 'SYMBOL.BO' (e.g., 'RELIANCE.BO')
        
        Returns:
            Last Traded Price as float, or None if error
        """
        try:
            ticker = yf.Ticker(symbol)
            hist = ticker.history(period="2d")
            
            if not hist.empty:
                return round(hist['Close'].iloc[-1], 2)
            return None
        except Exception as e:
            print(f"Error fetching LTP for {symbol}: {e}")
            return None
    
    def get_quote(self, symbol: str) -> Dict:
        """
        Get detailed quote data including LTP, OHLC, volume, etc.
        
        Args:
            symbol: Stock symbol in Yahoo Finance format
        
        Returns:
            Dictionary with detailed quote data
        """
        try:
            ticker = yf.Ticker(symbol)
            info = ticker.info
            hist = ticker.history(period="2d")
            
            if hist.empty:
                return {'error': 'No data available'}
            
            quote_data = {
                'symbol': symbol,
                'name': info.get('longName', symbol),
                'last_price': round(hist['Close'].iloc[-1], 2),
                'open': round(hist['Open'].iloc[-1], 2) if 'Open' in hist.columns else None,
                'high': round(hist['High'].iloc[-1], 2) if 'High' in hist.columns else None,
                'low': round(hist['Low'].iloc[-1], 2) if 'Low' in hist.columns else None,
                'close': round(hist['Close'].iloc[-1], 2),
                'volume': int(hist['Volume'].iloc[-1]) if 'Volume' in hist.columns else None,
                'currency': info.get('currency', 'INR'),
                'market_cap': info.get('marketCap'),
                'previous_close': info.get('previousClose'),
                'day_change': round(hist['Close'].iloc[-1] - hist['Open'].iloc[-1], 2) if len(hist) > 0 else None,
                'day_change_percent': round(((hist['Close'].iloc[-1] - hist['Open'].iloc[-1]) / hist['Open'].iloc[-1]) * 100, 2) if len(hist) > 0 and hist['Open'].iloc[-1] != 0 else None
            }
            
            return quote_data
        except Exception as e:
            return {'error': str(e)}
    
    def get_realtime_price(self, symbol: str) -> Optional[float]:
        """
        Get real-time or latest available price.
        
        Args:
            symbol: Stock symbol in Yahoo Finance format
        
        Returns:
            Current price as float, or None if error
        """
        try:
            ticker = yf.Ticker(symbol)
            data = ticker.history(period="2d", interval="1m")
            
            if not data.empty:
                return round(data['Close'].iloc[-1], 2)
            else:
                # Fallback to daily data
                return self.get_ltp_single(symbol)
        except Exception as e:
            print(f"Error fetching real-time price for {symbol}: {e}")
            return None


def main():
    """Example usage of StockLTP class."""
    
    # Initialize the StockLTP class
    stock_ltp = StockLTP()
    
    # Example 1: Get LTP for multiple stocks
    print("=== Getting LTP for multiple stocks ===")
    symbols = ['INFY.NS', 'SBIN.NS', 'TCS.NS', 'RELIANCE.NS']
    ltp_data = stock_ltp.get_ltp(symbols)
    
    for symbol, data in ltp_data.items():
        if 'error' in data:
            print(f"{symbol}: Error - {data['error']}")
        else:
            print(f"{symbol} ({data.get('name', 'N/A')}): ₹{data.get('last_price', 'N/A')}")
    
    print("\n=== Getting LTP for a single stock ===")
    # Example 2: Get LTP for a single stock
    ltp = stock_ltp.get_ltp_single('INFY.NS')
    print(f"INFY.NS LTP: ₹{ltp}")
    
    print("\n=== Getting detailed quote ===")
    # Example 3: Get detailed quote
    quote = stock_ltp.get_quote('INFY.NS')
    if 'error' not in quote:
        print(f"Symbol: {quote.get('symbol')}")
        print(f"Name: {quote.get('name')}")
        print(f"Last Price: ₹{quote.get('last_price')}")
        print(f"Open: ₹{quote.get('open')}")
        print(f"High: ₹{quote.get('high')}")
        print(f"Low: ₹{quote.get('low')}")
        print(f"Close: ₹{quote.get('close')}")
        print(f"Volume: {quote.get('volume')}")
        print(f"Day Change: ₹{quote.get('day_change')} ({quote.get('day_change_percent')}%)")
    else:
        print(f"Error: {quote.get('error')}")
    
    print("\n=== Popular Indian Stocks ===")
    # Example 4: Popular Indian stocks
    popular_stocks = {
        'NSE': ['NIFTYBEES.NS',
'INDUSINDBK.NS',
'ITC.NS',
'TMPV.NS',
'NATCOPHARM.NS',
'DRREDDY.NS',
'NEXT50IETF.NS',
'GOLDIETF.NS',
'TCS.NS',
'TATACAP.NS',
'KTKBANK.NS',
'JYOTHYLAB.NS',
'TMCV.NS',
'ZYDUSLIFE.NS',
'FINCABLES.NS',
'IDFCFIRSTB.NS',
'INFY.NS',
'AXISBANK.NS',
'WIPRO.NS',
'SPANDANA.NS',
'SOUTHBANK.NS',
'TMB.NS',
'HCLTECH.NS',
'PETRONET.NS',
'ITBEES.NS',
'YESBANK.NS',
'MANAPPURAM.NS',
'ABCAPITAL.NS',
'GOLDCASE.NS'],
        'BSE': ['RELIANCE.BO', 'TCS.BO']
    }
    
    for exchange, stocks in popular_stocks.items():
        print(f"\n{exchange} Stocks:")
        for stock in stocks:
            ltp = stock_ltp.get_ltp_single(stock)
            if ltp:
                print(f"  {stock}: ₹{ltp}")


if __name__ == "__main__":
    main()


=== Getting LTP for multiple stocks ===
INFY.NS (Infosys Limited): ₹1255.9
SBIN.NS (State Bank of India): ₹1058.0
TCS.NS (Tata Consultancy Services Limited): ₹2390.6
RELIANCE.NS (Reliance Industries Limited): ₹1414.4

=== Getting LTP for a single stock ===
INFY.NS LTP: ₹1255.9

=== Getting detailed quote ===
Symbol: INFY.NS
Name: Infosys Limited
Last Price: ₹1255.9
Open: ₹1242.0
High: ₹1264.8
Low: ₹1229.2
Close: ₹1255.9
Volume: 27296420
Day Change: ₹13.9 (1.12%)

=== Popular Indian Stocks ===

NSE Stocks:
  NIFTYBEES.NS: ₹262.07
  INDUSINDBK.NS: ₹818.6
  ITC.NS: ₹299.95
  TMPV.NS: ₹314.1
  NATCOPHARM.NS: ₹959.2
  DRREDDY.NS: ₹1298.9
  NEXT50IETF.NS: ₹67.3
  GOLDIETF.NS: ₹125.18
  TCS.NS: ₹2390.6
  TATACAP.NS: ₹317.1
  KTKBANK.NS: ₹228.93
  JYOTHYLAB.NS: ₹208.55
  TMCV.NS: ₹419.65
  ZYDUSLIFE.NS: ₹890.5
  FINCABLES.NS: ₹878.15
  IDFCFIRSTB.NS: ₹62.95
  INFY.NS: ₹1255.9
  AXISBANK.NS: ₹1203.9
  WIPRO.NS: ₹190.9
  SPANDANA.NS: ₹207.85
  SOUTHBANK.NS: ₹36.01
  TMB.NS: ₹599.0
  HCLTECH.NS: 

In [3]:
import yfinance as yf
popular_stocks = {
        'NSE': ['NIFTYBEES.NS',
'INDUSINDBK.NS',
'ITC.NS',
'TMPV.NS',
'NATCOPHARM.NS',
'DRREDDY.NS',
'NEXT50IETF.NS',
'GOLDIETF.NS',
'TCS.NS',
'TATACAP.NS',
'KTKBANK.NS',
'JYOTHYLAB.NS',
'TMCV.NS',
'ZYDUSLIFE.NS',
'FINCABLES.NS',
'IDFCFIRSTB.NS',
'INFY.NS',
'AXISBANK.NS',
'WIPRO.NS',
'SPANDANA.NS',
'SOUTHBANK.NS',
'TMB.NS',
'HCLTECH.NS',
'PETRONET.NS',
'ITBEES.NS',
'YESBANK.NS',
'HDFCBANK.NS',
'MANAPPURAM.NS',
'ABCAPITAL.NS',
'GOLDCASE.NS'],
        'BSE': ['RELIANCE.BO', 'TCS.BO']
    }
i = 0
for i in range(len(popular_stocks['NSE'])):
    ticker = yf.Ticker(popular_stocks['NSE'][i])
    info = ticker.info
    hist = ticker.history(period="21d")
    hist
    print(f"  {popular_stocks['NSE'][i]}: ₹{round(hist['Close'].iloc[-21],2)}")
    i += 1
     

  NIFTYBEES.NS: ₹288.38
  INDUSINDBK.NS: ₹927.35
  ITC.NS: ₹326.0
  TMPV.NS: ₹375.7
  NATCOPHARM.NS: ₹888.3
  DRREDDY.NS: ₹1282.3
  NEXT50IETF.NS: ₹72.66
  GOLDIETF.NS: ₹131.49
  TCS.NS: ₹2677.9
  TATACAP.NS: ₹348.3
  KTKBANK.NS: ₹204.53
  JYOTHYLAB.NS: ₹239.93
  TMCV.NS: ₹474.5
  ZYDUSLIFE.NS: ₹902.4
  FINCABLES.NS: ₹801.7
  IDFCFIRSTB.NS: ₹82.98
  INFY.NS: ₹1370.5
  AXISBANK.NS: ₹1356.6
  WIPRO.NS: ₹211.21
  SPANDANA.NS: ₹262.0
  SOUTHBANK.NS: ₹40.04
  TMB.NS: ₹685.7
  HCLTECH.NS: ₹1450.4
  PETRONET.NS: ₹301.8
  ITBEES.NS: ₹35.71
  YESBANK.NS: ₹20.96
  HDFCBANK.NS: ₹915.6
  MANAPPURAM.NS: ₹305.7
  ABCAPITAL.NS: ₹344.3
  GOLDCASE.NS: ₹24.18
